# 03 RQ2 Robustness Under Synthetic Perturbations

This notebook evaluates the robustness of the trained CNN backbone models under controlled synthetic perturbations.

It performs:
- checkpoint loading
- clean evaluation
- perturbation evaluation across multiple severity levels
- comparison across perturbation types
- export of tables and figures

Outputs:
- Table 3: average accuracy under increasing perturbation severity
- Table 4: perturbation-type comparison at highest severity
- Figure 3: robustness degradation curves
- Figure 4: comparative robustness across perturbation types
- ZIP archive of RQ2 outputs

In [1]:
# ----------------------------------------
# Section 1: Imports
# ----------------------------------------

import os
import json
import random
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

from PIL import Image, ImageFilter, ImageEnhance

from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [2]:
# ----------------------------------------
# Section 2: Reproducibility setup
# ----------------------------------------

SEED = 42

def seed_everything(seed: int = 42) -> None:
    """
    Set random seeds for reproducibility across Python, NumPy, and PyTorch.
    """
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id: int) -> None:
    """
    Ensure each DataLoader worker uses a deterministic seed.
    """
    worker_seed = SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)

seed_everything(SEED)

print("Reproducibility setup completed")
print(f"Global seed: {SEED}")

Reproducibility setup completed
Global seed: 42


In [9]:
# ----------------------------------------
# Section 3: Configuration
# ----------------------------------------

CONFIG = {
    "seed": SEED,
    "image_size": 224,
    "batch_size": 32,
    "num_workers": 0,
    "plantvillage_root": "/kaggle/input/datasets/thedataeng/plantvillage",
    "checkpoint_root": "/kaggle/input/datasets/thedataeng/thesis-train-backbones-outputs",
    "output_root": "/kaggle/working/thesis_outputs/rq2_robustness",
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT_ROOT = Path(CONFIG["output_root"])
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

MODEL_NAMES = ["resnet50", "efficientnet_b0", "mobilenet_v2"]
PERTURBATIONS = ["blur", "low_light", "occlusion", "background_clutter"]
SEVERITY_LEVELS = [1, 2, 3, 4, 5]

print("Configuration loaded")
print(f"Device: {DEVICE}")
print(f"Checkpoint root: {CONFIG['checkpoint_root']}")
print(f"Output root: {OUTPUT_ROOT}")

Configuration loaded
Device: cuda
Checkpoint root: /kaggle/input/datasets/thedataeng/thesis-train-backbones-outputs
Output root: /kaggle/working/thesis_outputs/rq2_robustness


In [10]:
# ----------------------------------------
# Section 4: Helper functions
# ----------------------------------------

def ensure_dir(path: Path) -> Path:
    """
    Create a directory if it does not exist and return the Path object.
    """
    path.mkdir(parents=True, exist_ok=True)
    return path

def get_eval_transform(image_size: int = 224):
    """
    Create the evaluation transform used across RQ2 experiments.
    """
    return transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        ),
    ])

def create_model(model_name: str, num_classes: int) -> nn.Module:
    """
    Create the selected backbone model and replace the classification head.
    """
    if model_name == "resnet50":
        model = models.resnet50(weights=None)
        model.fc = nn.Linear(model.fc.in_features, num_classes)

    elif model_name == "efficientnet_b0":
        model = models.efficientnet_b0(weights=None)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

    elif model_name == "mobilenet_v2":
        model = models.mobilenet_v2(weights=None)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

    else:
        raise ValueError(f"Unsupported model name: {model_name}")

    return model.to(DEVICE)

def checkpoint_path(model_name: str) -> Path:
    """
    Return the checkpoint path for a given model name.
    """
    return Path(CONFIG["checkpoint_root"]) / "checkpoints" / f"{model_name}_seed{SEED}_best.pt"

@torch.no_grad()
def predict_loader(model: nn.Module, loader: DataLoader):
    """
    Run model inference on a DataLoader and return probabilities, predictions, and labels.
    """
    model.eval()

    all_probs = []
    all_preds = []
    all_labels = []

    for images, labels in loader:
        images = images.to(DEVICE)

        logits = model(images)
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        preds = probs.argmax(axis=1)

        all_probs.append(probs)
        all_preds.append(preds)
        all_labels.append(labels.numpy())

    return (
        np.vstack(all_probs),
        np.concatenate(all_preds),
        np.concatenate(all_labels),
    )

def evaluate_model(model: nn.Module, loader: DataLoader) -> dict:
    """
    Evaluate the model and return standard classification metrics.
    """
    probs, preds, labels = predict_loader(model, loader)

    accuracy = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="macro",
        zero_division=0
    )

    return {
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "preds": preds,
        "labels": labels,
        "probs": probs,
    }

def pretty_metric(x: float) -> float:
    """
    Round a metric value for cleaner table presentation.
    """
    return round(float(x), 4)

def save_table(df: pd.DataFrame, name: str) -> None:
    """
    Save a DataFrame as CSV in the tables directory.
    """
    table_dir = ensure_dir(OUTPUT_ROOT / "tables")
    csv_path = table_dir / f"{name}.csv"
    df.to_csv(csv_path, index=False)
    print(f"Saved table: {csv_path}")

def save_figure(fig: plt.Figure, name: str) -> None:
    """
    Save a matplotlib figure as PDF in the figures directory.
    """
    fig_dir = ensure_dir(OUTPUT_ROOT / "figures")
    pdf_path = fig_dir / f"{name}.pdf"
    fig.tight_layout()
    fig.savefig(pdf_path, format="pdf", bbox_inches="tight")
    plt.close(fig)
    print(f"Saved figure: {pdf_path}")

print('Done')

Done


In [11]:
# ----------------------------------------
# Section 5: Perturbation dataset wrapper
# ----------------------------------------

class PerturbedImageFolder(datasets.ImageFolder):
    """
    Apply a synthetic perturbation to each image before the standard evaluation transform.
    """

    def __init__(self, root, base_transform, perturbation="blur", severity=1):
        super().__init__(root=root, transform=None)
        self.base_transform = base_transform
        self.perturbation = perturbation
        self.severity = severity

    def apply_perturbation(self, img: Image.Image) -> Image.Image:
        """
        Apply the selected perturbation at the given severity level.
        """
        if self.perturbation == "blur":
            return img.filter(ImageFilter.GaussianBlur(radius=self.severity))

        elif self.perturbation == "low_light":
            factor = max(0.15, 1.0 - 0.18 * self.severity)
            return ImageEnhance.Brightness(img).enhance(factor)

        elif self.perturbation == "occlusion":
            img = img.copy()
            arr = np.array(img)
            h, w = arr.shape[:2]

            occ = int(min(h, w) * (0.08 * self.severity + 0.04))
            x0 = w // 2 - occ // 2
            y0 = h // 2 - occ // 2

            arr[y0:y0 + occ, x0:x0 + occ] = 0
            return Image.fromarray(arr)

        elif self.perturbation == "background_clutter":
            img = img.copy()
            arr = np.array(img)
            h, w = arr.shape[:2]

            rng = np.random.default_rng(SEED + self.severity)
            for _ in range(6 * self.severity):
                x = rng.integers(0, w)
                y = rng.integers(0, h)
                rr = rng.integers(6, 18)

                y1 = max(0, y - rr)
                y2 = min(h, y + rr)
                x1 = max(0, x - rr)
                x2 = min(w, x + rr)

                arr[y1:y2, x1:x2] = rng.integers(
                    0,
                    255,
                    size=(y2 - y1, x2 - x1, 3),
                    dtype=np.uint8
                )

            return Image.fromarray(arr)

        return img

    def __getitem__(self, index):
        path, target = self.samples[index]
        sample = self.loader(path).convert("RGB")
        sample = self.apply_perturbation(sample)

        if self.base_transform is not None:
            sample = self.base_transform(sample)

        return sample, target

print('Done')

Done


In [12]:
# ----------------------------------------
# Section 6: Dataset loading
# ----------------------------------------

pv_root = Path(CONFIG["plantvillage_root"])
test_dir = pv_root / "test"

assert test_dir.exists(), f"Missing PlantVillage test directory: {test_dir}"

eval_tfms = get_eval_transform(CONFIG["image_size"])

test_dataset = datasets.ImageFolder(test_dir, transform=eval_tfms)

generator = torch.Generator()
generator.manual_seed(SEED)

test_loader = DataLoader(
    test_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    worker_init_fn=seed_worker,
    generator=generator,
    pin_memory=True,
)

CLASS_NAMES = test_dataset.classes
NUM_CLASSES = len(CLASS_NAMES)

print("PlantVillage test dataset loaded successfully")
print(f"Test samples:      {len(test_dataset)}")
print(f"Number of classes: {NUM_CLASSES}")

PlantVillage test dataset loaded successfully
Test samples:      5553
Number of classes: 27


In [13]:
# ----------------------------------------
# Section 7: Checkpoint loading
# ----------------------------------------

models_loaded = {}

for idx, model_name in enumerate(MODEL_NAMES, start=1):
    print(f"Loading checkpoint {idx}/{len(MODEL_NAMES)}: {model_name}")

    ckpt_path = checkpoint_path(model_name)
    assert ckpt_path.exists(), f"Missing checkpoint: {ckpt_path}"

    model = create_model(model_name, NUM_CLASSES)
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    model.eval()

    models_loaded[model_name] = model

    print(f"Loaded checkpoint: {ckpt_path}")

print("All checkpoints loaded successfully")

Loading checkpoint 1/3: resnet50
Loaded checkpoint: /kaggle/input/datasets/thedataeng/thesis-train-backbones-outputs/checkpoints/resnet50_seed42_best.pt
Loading checkpoint 2/3: efficientnet_b0
Loaded checkpoint: /kaggle/input/datasets/thedataeng/thesis-train-backbones-outputs/checkpoints/efficientnet_b0_seed42_best.pt
Loading checkpoint 3/3: mobilenet_v2
Loaded checkpoint: /kaggle/input/datasets/thedataeng/thesis-train-backbones-outputs/checkpoints/mobilenet_v2_seed42_best.pt
All checkpoints loaded successfully


In [14]:
# ----------------------------------------
# Section 8: Robustness evaluation
# ----------------------------------------

perturbation_rows = []

for idx, model_name in enumerate(MODEL_NAMES, start=1):
    print(f"Evaluating model {idx}/{len(MODEL_NAMES)}: {model_name}")

    model = models_loaded[model_name]

    # Evaluate on clean PlantVillage test set
    print("  Running clean evaluation")
    clean_metrics = evaluate_model(model, test_loader)

    perturbation_rows.append({
        "Model": model_name,
        "Perturbation": "clean",
        "Severity": 0,
        "Accuracy": pretty_metric(clean_metrics["accuracy"]),
        "Precision": pretty_metric(clean_metrics["precision"]),
        "Recall": pretty_metric(clean_metrics["recall"]),
        "F1-score": pretty_metric(clean_metrics["f1"]),
    })

    # Evaluate across perturbation types and severity levels
    for perturbation in PERTURBATIONS:
        print(f"  Running perturbation: {perturbation}")

        for severity in SEVERITY_LEVELS:
            print(f"    Severity level: {severity}")

            perturbed_dataset = PerturbedImageFolder(
                root=test_dir,
                base_transform=eval_tfms,
                perturbation=perturbation,
                severity=severity,
            )

            perturbed_loader = DataLoader(
                perturbed_dataset,
                batch_size=CONFIG["batch_size"],
                shuffle=False,
                num_workers=CONFIG["num_workers"],
                worker_init_fn=seed_worker,
                generator=generator,
                pin_memory=True,
            )

            metrics = evaluate_model(model, perturbed_loader)

            perturbation_rows.append({
                "Model": model_name,
                "Perturbation": perturbation,
                "Severity": severity,
                "Accuracy": pretty_metric(metrics["accuracy"]),
                "Precision": pretty_metric(metrics["precision"]),
                "Recall": pretty_metric(metrics["recall"]),
                "F1-score": pretty_metric(metrics["f1"]),
            })

print("Robustness evaluation completed successfully")

Evaluating model 1/3: resnet50
  Running clean evaluation
  Running perturbation: blur
    Severity level: 1
    Severity level: 2
    Severity level: 3
    Severity level: 4
    Severity level: 5
  Running perturbation: low_light
    Severity level: 1
    Severity level: 2
    Severity level: 3
    Severity level: 4
    Severity level: 5
  Running perturbation: occlusion
    Severity level: 1
    Severity level: 2
    Severity level: 3
    Severity level: 4
    Severity level: 5
  Running perturbation: background_clutter
    Severity level: 1
    Severity level: 2
    Severity level: 3
    Severity level: 4
    Severity level: 5
Evaluating model 2/3: efficientnet_b0
  Running clean evaluation
  Running perturbation: blur
    Severity level: 1
    Severity level: 2
    Severity level: 3
    Severity level: 4
    Severity level: 5
  Running perturbation: low_light
    Severity level: 1
    Severity level: 2
    Severity level: 3
    Severity level: 4
    Severity level: 5
  Running pert

In [15]:
# ----------------------------------------
# Section 9: Save raw perturbation results
# ----------------------------------------

perturbation_df = pd.DataFrame(perturbation_rows)

pretty_names = {
    "resnet50": "ResNet50",
    "efficientnet_b0": "EfficientNet-B0",
    "mobilenet_v2": "MobileNetV2",
}
perturbation_df["Model"] = perturbation_df["Model"].map(pretty_names)

save_table(perturbation_df, "Raw_Perturbation_Robustness_Results")

print("Raw robustness results saved successfully")
display(perturbation_df.head(10))

Saved table: /kaggle/working/thesis_outputs/rq2_robustness/tables/Raw_Perturbation_Robustness_Results.csv
Raw robustness results saved successfully


,Model,Perturbation,Severity,Accuracy,Precision,Recall,F1-score
0,ResNet50,clean,0,0.9950,0.9912,0.9931,0.9921
1,ResNet50,blur,1,0.9797,0.9784,0.9820,0.9789
2,ResNet50,blur,2,0.8613,0.9156,0.8698,0.8669
3,ResNet50,blur,3,0.6469,0.8359,0.5877,0.5961
4,ResNet50,blur,4,0.4380,0.5958,0.2945,0.2793
5,ResNet50,blur,5,0.3366,0.2540,0.1708,0.1332
6,ResNet50,low_light,1,0.9937,0.9898,0.9915,0.9905
7,ResNet50,low_light,2,0.9932,0.9897,0.9920,0.9907
8,ResNet50,low_light,3,0.9879,0.9848,0.9863,0.9852
9,ResNet50,low_light,4,0.9647,0.9656,0.9620,0.9613


In [16]:
# ----------------------------------------
# Section 10: Save Table 3 - Average accuracy by severity
# ----------------------------------------

severity_table = (
    perturbation_df.groupby(["Model", "Severity"])["Accuracy"]
    .mean()
    .reset_index()
    .pivot(index="Severity", columns="Model", values="Accuracy")
    .reset_index()
)

severity_table.columns.name = None
severity_table = severity_table.rename_axis(None, axis=1)

save_table(severity_table, "Table_3_Average_Accuracy_By_Severity")

print("Table 3 saved successfully")
display(severity_table)

Saved table: /kaggle/working/thesis_outputs/rq2_robustness/tables/Table_3_Average_Accuracy_By_Severity.csv
Table 3 saved successfully


,Severity,EfficientNet-B0,MobileNetV2,ResNet50
0,0,0.996200,0.994800,0.995000
1,1,0.993075,0.981625,0.989150
2,2,0.948175,0.916425,0.958050
3,3,0.858275,0.814975,0.899300
4,4,0.818700,0.694150,0.800525
5,5,0.717250,0.658700,0.759475


In [17]:
# ----------------------------------------
# Section 11: Save Table 4 - Perturbation comparison at severity 5
# ----------------------------------------

severity_5_df = perturbation_df[
    (perturbation_df["Severity"] == 5) &
    (perturbation_df["Perturbation"] != "clean")
].copy()

table4 = (
    severity_5_df.pivot(index="Perturbation", columns="Model", values="Accuracy")
    .reset_index()
)

table4["Approx. Mean"] = table4[["ResNet50", "EfficientNet-B0", "MobileNetV2"]].mean(axis=1).round(4)

save_table(table4, "Table_4_Perturbation_Type_Comparison_Severity_5")

print("Table 4 saved successfully")
display(table4)

Saved table: /kaggle/working/thesis_outputs/rq2_robustness/tables/Table_4_Perturbation_Type_Comparison_Severity_5.csv
Table 4 saved successfully


Model,Perturbation,EfficientNet-B0,MobileNetV2,ResNet50,Approx. Mean
0,background_clutter,0.9744,0.7382,0.9634,0.8920
1,blur,0.3052,0.2431,0.3366,0.2950
2,low_light,0.6179,0.7578,0.7835,0.7197
3,occlusion,0.9715,0.8957,0.9544,0.9405


In [18]:
# ----------------------------------------
# Section 12: Figure 3 - Robustness degradation curves
# ----------------------------------------

fig, ax = plt.subplots(figsize=(8.5, 5))

severity_curve_df = perturbation_df[
    perturbation_df["Perturbation"] != "clean"
].groupby(["Model", "Severity"])["Accuracy"].mean().reset_index()

for model_name in ["ResNet50", "EfficientNet-B0", "MobileNetV2"]:
    subset = severity_curve_df[severity_curve_df["Model"] == model_name]
    ax.plot(
        subset["Severity"],
        subset["Accuracy"],
        marker="o",
        linewidth=2,
        label=model_name
    )

ax.set_xlabel("Perturbation severity")
ax.set_ylabel("Average accuracy")
ax.set_title("Figure 3. Robustness Degradation Under Increasing Perturbation Severity")
ax.grid(alpha=0.25)
ax.legend(frameon=True)

save_figure(fig, "Figure_3_Robustness_Degradation")

Saved figure: /kaggle/working/thesis_outputs/rq2_robustness/figures/Figure_3_Robustness_Degradation.pdf


In [19]:
# ----------------------------------------
# Section 13: Figure 4 - Perturbation type comparison
# ----------------------------------------

fig, ax = plt.subplots(figsize=(9, 5))

plot_df = severity_5_df.copy()
perturbation_order = ["blur", "low_light", "occlusion", "background_clutter"]
model_order = ["ResNet50", "EfficientNet-B0", "MobileNetV2"]

x = np.arange(len(perturbation_order))
width = 0.23

for i, model_name in enumerate(model_order):
    values = []
    for perturbation in perturbation_order:
        row = plot_df[
            (plot_df["Model"] == model_name) &
            (plot_df["Perturbation"] == perturbation)
        ].iloc[0]
        values.append(row["Accuracy"])

    ax.bar(x + (i - 1) * width, values, width=width, label=model_name)

ax.set_xticks(x)
ax.set_xticklabels(["Blur", "Low Light", "Occlusion", "Background Clutter"], rotation=10)
ax.set_ylabel("Accuracy at severity 5")
ax.set_title("Figure 4. Comparative Robustness Across Synthetic Perturbation Types")
ax.grid(axis="y", alpha=0.25)
ax.legend(frameon=True)

save_figure(fig, "Figure_4_Perturbation_Type_Comparison")

Saved figure: /kaggle/working/thesis_outputs/rq2_robustness/figures/Figure_4_Perturbation_Type_Comparison.pdf


In [20]:
# ----------------------------------------
# Section 14: Save RQ2 metadata
# ----------------------------------------

meta_dir = ensure_dir(OUTPUT_ROOT / "metadata")

rq2_meta = {
    "seed": SEED,
    "models_evaluated": MODEL_NAMES,
    "perturbations": PERTURBATIONS,
    "severity_levels": SEVERITY_LEVELS,
    "num_classes": NUM_CLASSES,
}

meta_path = meta_dir / "rq2_metadata.json"
with open(meta_path, "w") as f:
    json.dump(rq2_meta, f, indent=2)

print("RQ2 metadata saved successfully")
print(f"Metadata path: {meta_path}")

RQ2 metadata saved successfully
Metadata path: /kaggle/working/thesis_outputs/rq2_robustness/metadata/rq2_metadata.json


In [21]:
# ----------------------------------------
# Section 15: Create ZIP archive
# ----------------------------------------

zip_path = OUTPUT_ROOT.parent / "03_rq2_robustness_outputs.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for file_path in OUTPUT_ROOT.rglob("*"):
        if file_path.is_file():
            zf.write(file_path, arcname=file_path.relative_to(OUTPUT_ROOT))

print("ZIP archive created successfully")
print(f"ZIP file: {zip_path}")
print("03_rq2_robustness notebook completed successfully")

ZIP archive created successfully
ZIP file: /kaggle/working/thesis_outputs/03_rq2_robustness_outputs.zip
03_rq2_robustness notebook completed successfully
